# Day 3: RoPE 与 SwiGLU 的形象实验

选择 Python 3 (ipykernel)，从上到下执行。使用 CPU，不下载模型，不训练参数。
RoPE 属于 Attention 内部，gate/up/down + SiLU 属于 FFN；先分开观察。
配套阅读：`docs/ROPE_SWIGLU_INTUITION.md`。图中用英文标签，避免容器缺少中文字体。

In [ ]:
import math
import os
import sys
from pathlib import Path

root = Path.cwd()
if root.name == "learning":
    root = root.parent
if not (root / "reasoning_from_scratch/qwen3.py").is_file():
    raise RuntimeError("请从仓库根目录或 learning 目录启动。")
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import torch
import matplotlib.pyplot as plt
from torch.nn import functional as F
from reasoning_from_scratch.qwen3 import FeedForward, apply_rope, compute_rope_params

## 1. 把一对特征看成平面箭头

先固定内容 `[1,0]`，人为设置每个位置转 30 度。观察同一个箭头的位置相位如何变化，长度没有变化。
**这是旋转示意，不是 Qwen3 实际的频率设置。** 实际 RoPE 用多个不同频率旋转多对特征。

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
for position, color in zip((0, 1, 2), ("#137547", "#cf3e36", "#2567bd")):
    angle = position * math.pi / 6
    a, b = math.cos(angle), math.sin(angle)
    ax.annotate("", xy=(a, b), xytext=(0, 0),
                arrowprops={"arrowstyle": "->", "color": color, "lw": 2.5})
    label_y = b * 1.1 + (0.12 if position == 0 else 0)
    ax.text(a * 1.1, label_y, f"pos {position}: {position * 30} deg",
            color=color, fontsize=10, ha="center")
    print(f"position={position}: [{a:.4f}, {b:.4f}], length={math.hypot(a, b):.4f}")
ax.axhline(0, color="#999999", lw=0.8)
ax.axvline(0, color="#999999", lw=0.8)
ax.set(xlim=(-0.3, 1.6), ylim=(-0.3, 1.3), aspect="equal",
       xlabel="feature a", ylabel="feature b", title="Same content, different position phase")
ax.grid(alpha=0.2)
plt.show()

## 2. 调用仓库 RoPE，验证相对位置

固定未旋转的 q/k 内容，比较位置 2/5 与 7/10：整体平移了 5，相对距离仍为 3。
当前仓库的 4 维实现按 `(0,2)`、`(1,3)` 配对，不是相邻维度配对。
这个等价性针对固定的 q/k 和相同频率设置，不意味着真实模型里相同距离的词总有相同分数。

In [ ]:
cos, sin = compute_rope_params(head_dim=4, context_length=16, theta_base=10_000)
q = torch.tensor([1.0, 2.0, 3.0, 4.0]).reshape(1, 1, 1, 4)
k = torch.tensor([2.0, -1.0, 0.5, 3.0]).reshape(1, 1, 1, 4)
q2, k5 = apply_rope(q, cos, sin, offset=2), apply_rope(k, cos, sin, offset=5)
q7, k10 = apply_rope(q, cos, sin, offset=7), apply_rope(k, cos, sin, offset=10)
score_2_5, score_7_10 = (q2 * k5).sum(), (q7 * k10).sum()
torch.testing.assert_close(q.norm(), q2.norm(), atol=1e-5, rtol=1e-5)
torch.testing.assert_close(score_2_5, score_7_10, atol=1e-5, rtol=1e-5)
print("shape before/after:", tuple(q.shape), tuple(q2.shape))
print("length before/after:", q.norm().item(), q2.norm().item())
print("dot at positions 2/5:", score_2_5.item())
print("dot at positions 7/10:", score_7_10.item())
print("relative-position and length checks: passed")

## 3. SiLU 不是 sigmoid

`SiLU(z) = z * sigmoid(z)`。橙色 sigmoid 曲线在 0 到 1 之间，但蓝色 SiLU 可以为负或大于 1。
所以 SwiGLU 中的 gate 不是概率，也不是只能削弱特征的开关。

In [ ]:
z = torch.linspace(-5, 5, 300)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(z.numpy(), F.silu(z).numpy(), label="SiLU(z) = z * sigmoid(z)", color="#2567bd", lw=2)
ax.plot(z.numpy(), torch.sigmoid(z).numpy(), label="sigmoid(z)", color="#c06c18", lw=2)
ax.axhline(0, color="#999999", lw=0.8)
ax.axvline(0, color="#999999", lw=0.8)
ax.set(xlabel="z", ylabel="output", title="SiLU can be negative or greater than one")
ax.legend()
ax.grid(alpha=0.2)
plt.show()
for value in (-2.0, 0.0, 2.0):
    t = torch.tensor(value)
    print(f"z={value:4.1f}, sigmoid={t.sigmoid().item():.4f}, SiLU={F.silu(t).item():.4f}")

## 4. 同一输入的两个分支，最后重新组合

将真实 FFN 缩为 `2 → 3 → 2`，手动设置权重来说明数据流，不代表真实模型学到这些权重。
这里 gate 产生 `[-2,0,2]`，up 产生 `[10,10,10]`；先看 SiLU 和逐元素乘法，再看 down。
仓库对应为 `fc1=gate_proj`、`fc2=up_proj`、`fc3=down_proj`。可以在每一行左侧设断点观察。

In [ ]:
ff = FeedForward({"emb_dim": 2, "hidden_dim": 3, "dtype": torch.float32}).eval()
with torch.no_grad():
    ff.fc1.weight.copy_(torch.tensor([[-2.0, 0.0], [0.0, 0.0], [2.0, 0.0]]))
    ff.fc2.weight.copy_(torch.tensor([[2.0, 4.0], [0.0, 5.0], [6.0, 2.0]]))
    ff.fc3.weight.copy_(torch.tensor([[1.0, 1.0, 0.0], [0.0, 1.0, 1.0]]))
with torch.inference_mode():
    x = torch.tensor([[[1.0, 2.0]]])
    gate = ff.fc1(x)
    up = ff.fc2(x)
    coefficients = F.silu(gate)
    gated = coefficients * up
    output = ff.fc3(gated)
    torch.testing.assert_close(output, ff(x))
for name, value in (("x", x), ("gate", gate), ("up", up), ("SiLU(gate)", coefficients),
                    ("product", gated), ("down output", output)):
    print(f"{name:12} shape={tuple(value.shape)} values={value.flatten().tolist()}")
print("residual x + FFN(x):", (x + output).flatten().tolist())
print("repository FFN alignment: passed")

## 自测

1. 为什么 RoPE 后 shape 不变，数值却变了？
2. 改成 q 的位置 2、k 的位置 6，相对位置实验还应与 7/10 相等吗？
3. gate 和 up 是前后串联吗？
4. SiLU(gate) 可以是负数或大于 1，这对上分支的特征有什么影响？
5. down 的矩阵怎样重新组合三个通道？它是在简单截断吗？

改动示例数值前先预测结果，再重新执行。这里不测性能，不需要 GPU。